## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [15]:
import subprocess
import shutil

if shutil.which("tippecanoe") is None:
    print("tippecanoe is not installed or not on PATH.")
    print("On macOS, install it with: brew install tippecanoe")
else:
    result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
    print(result.stdout or result.stderr)


tippecanoe is not installed or not on PATH.
On macOS, install it with: brew install tippecanoe


## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [16]:
from pathlib import Path
import subprocess
import time
import shutil

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

if shutil.which("tippecanoe") is None:
    print("tippecanoe is not installed or not on PATH.")
    print("Install with: brew install tippecanoe")
elif not input_file.exists():
    print(f"Input file not found: {input_file.resolve()}")
else:
    t0 = time.perf_counter()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.perf_counter() - t0

    print(result.stderr)   # tippecanoe writes progress to stderr
    print(f"\nCompleted in {elapsed:.1f}s")
    print(f"Output file: {output_file.resolve()}")


tippecanoe is not installed or not on PATH.
Install with: brew install tippecanoe


## Inspecting the Output

In [17]:
if output_file.exists() and input_file.exists():
    size_mb = output_file.stat().st_size / 1_000_000
    raw_mb  = input_file.stat().st_size  / 1_000_000

    print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
    print(f"Output (PMTiles):      {size_mb:.2f} MB")
    print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")
else:
    print("Run the previous tippecanoe cell first, or check that the input data file exists.")


Run the previous tippecanoe cell first, or check that the input data file exists.


## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [18]:
# Use tippecanoe's companion tools to show metadata, if available.
import shutil

inspect_file = output_file.with_suffix('.inspect.pmtiles')

if not output_file.exists():
    print("PMTiles output not found. Run the tippecanoe command first.")
else:
    if shutil.which("tile-join"):
        result = subprocess.run(
            ["tile-join", "--no-tile-compression", "--if-matched",
             f"--output={inspect_file}", str(output_file)],
            capture_output=True, text=True
        )
        print(result.stderr or result.stdout or "tile-join completed.")
    else:
        print("tile-join not found; skipping tile-join inspection.")

    if shutil.which("pmtiles"):
        result2 = subprocess.run(
            ["pmtiles", "show", str(output_file)],
            capture_output=True, text=True
        )
        print(result2.stdout or result2.stderr)
    else:
        print("pmtiles CLI not found. Optional install: pip install pmtiles")


PMTiles output not found. Run the tippecanoe command first.


## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [19]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

localtileserver not installed.
Install with: pip install localtileserver

Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.
File location: /workspaces/ricardoayala2510-Spatial-Data-Mapping/assigments completed/03-Data_Manager/data/railroads.pmtiles


## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [20]:
# Run tippecanoe with --maximum-zoom=8 and compare output size
from pathlib import Path
import subprocess
import time
import shutil

input_file = Path("../../data/ne_10m_railroads.geojson")
output_z14 = Path("../../data/railroads.pmtiles")
output_z8  = Path("../../data/railroads_z8.pmtiles")

cmd_z8 = [
    "tippecanoe",
    f"--output={output_z8}",
    "--force",
    "--minimum-zoom=1",
    "--maximum-zoom=8",
    "--simplification=10",
    "--drop-densest-as-needed",
    "--layer=railroads",
    str(input_file),
]

if shutil.which("tippecanoe") is None:
    print("tippecanoe is not installed or not on PATH.")
    print("Install with: brew install tippecanoe")
elif not input_file.exists():
    print(f"Input file not found: {input_file.resolve()}")
else:
    t0 = time.perf_counter()
    result = subprocess.run(cmd_z8, capture_output=True, text=True)
    elapsed = time.perf_counter() - t0

    print(result.stderr)
    print(f"Completed z8 build in {elapsed:.1f}s")

    if output_z8.exists():
        z8_mb = output_z8.stat().st_size / 1_000_000
        print(f"maxzoom 8 PMTiles size:  {z8_mb:.2f} MB")

        if output_z14.exists():
            z14_mb = output_z14.stat().st_size / 1_000_000
            print(f"maxzoom 14 PMTiles size: {z14_mb:.2f} MB")
            print(f"Size saved by limiting to z8: {z14_mb - z8_mb:.2f} MB")
            print(f"z8 file is {z8_mb / z14_mb:.2%} of the z14 file size")
        else:
            print("The z14 file does not exist yet. Run the original tippecanoe command above to compare.")

print()
print("Answer:")
print("Limiting max zoom to 8 saves storage and reduces the number of detailed tiles that must be generated.")
print("The cost is user experience at close zoom levels: railroads will look generalized or may disappear,")
print("because the tile pyramid stops before neighborhood-level detail is available.")


tippecanoe is not installed or not on PATH.
Install with: brew install tippecanoe

Answer:
Limiting max zoom to 8 saves storage and reduces the number of detailed tiles that must be generated.
The cost is user experience at close zoom levels: railroads will look generalized or may disappear,
because the tile pyramid stops before neighborhood-level detail is available.


## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [21]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices
EARTH_CIRCUMFERENCE_M = 40_075_016.686  # Web Mercator world width at equator
TILE_EXTENT = 4096                      # default vector-tile coordinate grid
SIMPLIFICATION_PIXELS = 10              # --simplification=10

module02_epsilons_deg = {
    "coarse": 1.0,
    "medium": 0.1,
    "fine": 0.01,
    "extra_fine": 0.001,
}

print("Tippecanoe --simplification=10, approximate ground tolerance at equator")
print(f"{'Zoom':>4} {'Tile width (m)':>16} {'m / tile pixel':>16} {'10 px tolerance (m)':>22}")
print("-" * 66)

for zoom in [2, 5, 8, 12]:
    tile_width_m = EARTH_CIRCUMFERENCE_M / (2 ** zoom)
    meters_per_tile_pixel = tile_width_m / TILE_EXTENT
    tolerance_m = meters_per_tile_pixel * SIMPLIFICATION_PIXELS
    print(f"{zoom:>4} {tile_width_m:>16,.1f} {meters_per_tile_pixel:>16,.2f} {tolerance_m:>22,.1f}")

print()
print("Our Module 02 degree-based epsilons, very rough meters using 1 degree ≈ 111 km:")
print(f"{'LOD':<12} {'epsilon (deg)':>14} {'rough meters':>16}")
print("-" * 44)
for lod, eps in module02_epsilons_deg.items():
    print(f"{lod:<12} {eps:>14g} {eps * 111_000:>16,.1f}")

print()
print("Comparison:")
print("Tippecanoe's tolerance automatically changes with zoom and tile resolution.")
print("Our Module 02 values were fixed degree tolerances, so they were easier to understand")
print("but less tied to the actual number of screen or tile pixels seen by the user.")


Tippecanoe --simplification=10, approximate ground tolerance at equator
Zoom   Tile width (m)   m / tile pixel    10 px tolerance (m)
------------------------------------------------------------------
   2     10,018,754.2         2,445.98               24,459.8
   5      1,252,344.3           305.75                3,057.5
   8        156,543.0            38.22                  382.2
  12          9,783.9             2.39                   23.9

Our Module 02 degree-based epsilons, very rough meters using 1 degree ≈ 111 km:
LOD           epsilon (deg)     rough meters
--------------------------------------------
coarse                    1        111,000.0
medium                  0.1         11,100.0
fine                   0.01          1,110.0
extra_fine            0.001            111.0

Comparison:
Tippecanoe's tolerance automatically changes with zoom and tile resolution.
Our Module 02 values were fixed degree tolerances, so they were easier to understand
but less tied to the actua

## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

---

**Check Your Understanding Answer**

`--drop-densest-as-needed` is a density-based rule. If tiles are too large, tippecanoe increases the spacing between features and drops the features that are least visible in dense areas at that zoom level. That is more adaptive than our manual `scalerank <= 4` filter because it reacts to the actual tile density and zoom level instead of using one global property cutoff everywhere. The tradeoff is control and predictability: our `scalerank` filter is easier to explain because we know exactly which features are allowed through, but tippecanoe's approach usually gives a better-looking production map because it drops detail only where the tile would otherwise become too heavy.

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?